In [6]:
import pandas as pd
import os
import numpy as np
from dotenv import load_dotenv
from elasticsearch import Elasticsearch
from elasticsearch.helpers import scan
from sklearn.cluster import KMeans


load_dotenv()  # Load API key from .env file

INDEX_NAME = "vectorhood"
OS_HOST = os.getenv("OS_HOST")
OS_API_KEY = os.getenv("OS_API_KEY")

client = Elasticsearch(
    hosts=[OS_HOST],
    api_key=OS_API_KEY
)


Use KMeans to find embedding groups

Maybe there are multi dimnstional features that are captured by the embeddings?
We can try and use KMeans clustering to find out if there are clusters of words that are similar to each other.


In [7]:
# Build an embedding matrix
docs = []
for doc in scan(client, index=INDEX_NAME, query={
    "query": { "match_all": {} },
    }):
    source = doc["_source"]
    docs.append({
        "name": source.get("name", ""),
        "category": source.get("category", ""),
        "embedding": source["embedding"]
    })

docs_without_sentences = [doc for doc in docs if doc["category"].strip() != "Sentence"]

print("Number of documents in the index:", len(docs))
print("Number of documents without sentences:", len(docs_without_sentences))

# Convert to DataFrame
df = pd.DataFrame(docs)
embedding_matrix = np.array(df["embedding"].tolist())


Number of documents in the index: 387
Number of documents without sentences: 387


In [8]:
# Perform KMeans clustering 

model = KMeans(n_clusters=4, max_iter=1000, n_init=100, random_state=42)
labels = model.fit_predict(embedding_matrix)
print(model.cluster_centers_)


[[-0.04169549  0.02589677  0.01869419 ...  0.00504117 -0.00722848
   0.01916864]
 [-0.04701867  0.02218765  0.0225435  ...  0.02651691 -0.00787397
   0.03295561]
 [-0.01456926  0.0424461   0.0215055  ... -0.00759765 -0.02350787
   0.0354111 ]
 [-0.02504796  0.03732977  0.04001699 ...  0.0072042   0.00031403
   0.02622988]]


In [9]:
# Now we can try and see for each cluster center, if it is surrounded by similar items

for i, center in enumerate(model.cluster_centers_):
    query_vector = center.tolist()
    body = {
        "knn": {
            "field": "embedding",
            "query_vector": query_vector,
            "k": 10,
            "num_candidates": 600
        }
    }
    response = client.search(index=INDEX_NAME, body=body)
    print(f"Cluster {i}:")
    for doc in response['hits']['hits']:
        source = doc['_source']
        print(f"  - Name: {source['name']}, Category: {source['category']}, Score: {doc['_score']}")

Cluster 0:
  - Name: Chili Pepper, Category: Vegetable, Score: 0.93120646
  - Name: Dish Soap, Category: Household, Score: 0.9294467
  - Name: Extension Cord, Category: Household, Score: 0.927696
  - Name: Compression Socks, Category: Clothing, Score: 0.9274373
  - Name: Oven Mitts, Category: Household, Score: 0.92578983
  - Name: Sweet Potato, Category: Vegetable, Score: 0.9215176
  - Name: Thermal Underwear, Category: Clothing, Score: 0.92145824
  - Name: Puffer Jacket, Category: Clothing, Score: 0.92143536
  - Name: Hair Dryer, Category: Household, Score: 0.9208689
  - Name: Bell Pepper, Category: Vegetable, Score: 0.920804
Cluster 1:
  - Name: Millipede, Category: Animal, Score: 0.9304817
  - Name: Avocado, Category: Fruit, Score: 0.93045855
  - Name: Spinach, Category: Vegetable, Score: 0.9287319
  - Name: Beetroot, Category: Vegetable, Score: 0.9284847
  - Name: Pomegranate, Category: Fruit, Score: 0.92640495
  - Name: Blueberry, Category: Fruit, Score: 0.92627335
  - Name: Raspb

Not really!